# 03-phase4-per-channel-alpha

**neuron Phase 4** — Phase 3 확정 sweet spot (scalar α_init=0.10) 의 **per-channel α 확장** 검증.

- **Phase 3 의 결과** (PR #46): scalar α=0.10, 3 시드 mean **1.7763 ± 0.0070**, GraphLM 대비 +0.0108 우위.
- **본 Phase 의 가설**:
  1. **per-channel α 의 표현력 우위** — α ∈ ℝ^{hidden_dim} 가 scalar α 보다 final_loss 개선?
  2. **partial-dead 패턴** — 채널 단위로 일부만 죽고 일부는 사는 patterns 존재? (지금까지는 0/100% binary 만)
  3. **paradigm 일관성** — Phase 2/3 의 sweet spot α_init=0.10 이 모든 채널 균일 init 으로 자연 확장?
- **설계**: 2 × 3 sweep (α_type ∈ {scalar, per-channel} × seed ∈ {42, 123, 456}) = 6 run, GPU 약 9분.
- **데이터**: TinyShakespeare (char-LM)
- **작성일**: 2026-05-25
- **연관**: Issue [#47](https://github.com/EinSofINTEREST/GraphLM/issues/47) / Phase 3 baseline PR [#46](https://github.com/EinSofINTEREST/GraphLM/pull/46)

## 1. 환경 / device

In [ ]:
import logging
import sys

import torch
import torch.nn.functional as F

import graphlm
from graphlm.data.tinyshakespeare import (
    CharTokenizer,
    TinyShakespeareDataset,
    iter_random_batches,
    load_tinyshakespeare_text,
)
from graphlm.neuron import NeuronConfig, NeuronGrowingDecoder, add_attn_smooth_start
from graphlm.training.triggers import PlateauTrigger
from graphlm.utils import repo_root, set_seed

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

logging.basicConfig(
    level=logging.WARNING, format="%(asctime)s %(levelname)s %(message)s", datefmt="%H:%M:%S"
)

print("python  :", sys.version.split()[0])
print("graphlm :", graphlm.__version__)
print("torch   :", torch.__version__)
print("device  :", DEVICE)
if str(DEVICE).startswith("cuda"):
    print(f"  device 0      : {torch.cuda.get_device_name(0)}")

## 2. 실험 설정 — 2 × 3 sweep

In [ ]:
ROOT = repo_root()
DATA_PATH = ROOT / "data" / "tinyshakespeare.txt"
OUT_DIR = ROOT / "runs" / "notebook-neuron-phase4"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEEDS = [42, 123, 456]
ALPHA_TYPES = ["scalar", "per_channel"]  # Phase 3 baseline + Phase 4 신규
ALPHA_INIT = 0.10  # Phase 2/3 sweet spot 고정

BATCH_SIZE = 16
BLOCK_SIZE = 128
BACKBONE = dict(hidden_dim=256, n_heads=4, ffn_dim=1024, n_layers=4, n_init_attn=1)
TRAIN = dict(
    lr=3e-4,
    max_steps=1500,
    max_total_attn=8,
    trigger_window=100,
    trigger_epsilon=0.04,
    trigger_cooldown=150,
    trigger_min_history=100,
)
DEAD_THRESHOLD = 0.05  # |α_channel| < 0.05 → channel dead (per-channel)
# scalar 의 경우 |α| < 0.05 → 전체 attn dead

# Phase 3 baseline (PR #46 결과 — scalar α=0.10 mean ± σ)
PHASE3_SCALAR_010_MEAN = 1.7763
PHASE3_SCALAR_010_SIGMA = 0.0070
GRAPHLM_PHASE1_4L_FINAL_LOSS = 1.7871

print(f"SEEDS        = {SEEDS}")
print(f"ALPHA_TYPES  = {ALPHA_TYPES}")
print(f"ALPHA_INIT   = {ALPHA_INIT} (Phase 2/3 sweet spot 고정)")
print(f"총 run       = {len(SEEDS) * len(ALPHA_TYPES)} (GPU 약 9분 예상)")

## 3. 데이터 로드

In [ ]:
text = load_tinyshakespeare_text(DATA_PATH)
tokenizer = CharTokenizer(text)
dataset = TinyShakespeareDataset(text, tokenizer)
print(f"vocab_size : {tokenizer.vocab_size}, n_tokens : {len(dataset):,}")

## 4. 2 × 3 sweep 학습

각 (α_type, seed) 조합마다 fresh model + train 1500 step + round-robin attn add.
`alpha_per_channel` 만 다르고 α_init / weight init / 데이터 iter / trigger 모두 동일.

In [ ]:
# {(alpha_type, seed): {'losses': [...], 'grow_events': [...], 'final_alphas': [...]}}
# final_alphas: per-channel 케이스에서는 각 entry 가 (bi, ai, tensor[hidden_dim])
runs = {}
for alpha_type in ALPHA_TYPES:
    per_channel = alpha_type == "per_channel"
    cfg = NeuronConfig(
        vocab_size=tokenizer.vocab_size,
        max_seq_len=BLOCK_SIZE,
        alpha_per_channel=per_channel,
        **BACKBONE,
    )
    for seed in SEEDS:
        print(f"--- alpha_type={alpha_type}, seed={seed} ---")
        set_seed(seed)
        model = NeuronGrowingDecoder(cfg).to(DEVICE)
        data_iter = iter_random_batches(
            dataset, batch_size=BATCH_SIZE, block_size=BLOCK_SIZE, seed=seed
        )
        trigger = PlateauTrigger(
            window=TRAIN["trigger_window"],
            epsilon=TRAIN["trigger_epsilon"],
            cooldown=TRAIN["trigger_cooldown"],
            min_history=TRAIN["trigger_min_history"],
        )
        optimizer = torch.optim.AdamW(model.parameters(), lr=TRAIN["lr"])

        losses = []
        grow_events = []
        next_block_to_grow = 0
        V = cfg.vocab_size
        model.train()
        for step in range(1, TRAIN["max_steps"] + 1):
            x, y = next(data_iter)
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            logits = model(x)
            loss = F.cross_entropy(logits.reshape(-1, V), y.reshape(-1))
            loss.backward()
            optimizer.step()
            losses.append(loss.item())

            if (
                trigger.update(loss.item())
                and sum(model.n_attn_per_block) < TRAIN["max_total_attn"]
            ):
                target_block = next_block_to_grow
                add_attn_smooth_start(model, target_block, alpha_init=ALPHA_INIT)
                block = model.blocks[target_block]
                new_params = (
                    list(block.attn_lns[-1].parameters())
                    + list(block.attns[-1].parameters())
                    + [block.attn_alphas[-1]]
                )
                optimizer.add_param_group({"params": new_params, "lr": TRAIN["lr"]})
                grow_events.append((step, target_block, sum(model.n_attn_per_block)))
                next_block_to_grow = (next_block_to_grow + 1) % cfg.n_layers

        # per-channel: tensor (hidden_dim,) 그대로 보관 / scalar: shape () 의 tensor → item()
        final_alphas = [
            (bi, ai, alpha_param.detach().cpu().clone())
            for bi, block in enumerate(model.blocks)
            for ai, alpha_param in enumerate(block.attn_alphas)
        ]
        runs[(alpha_type, seed)] = {
            "losses": losses,
            "grow_events": grow_events,
            "final_alphas": final_alphas,
        }
        n_last = min(100, len(losses))
        final_loss = sum(losses[-n_last:]) / n_last if n_last > 0 else 0.0
        print(f"  done: final_loss(last 100 avg)={final_loss:.4f}, n_added={len(grow_events)}")

        del model, optimizer, trigger, data_iter
        if str(DEVICE).startswith("cuda"):
            torch.cuda.empty_cache()

## 5. 결과 표 — α_type × seed

dead 판정 규칙:
- **scalar**: |α_scalar| < 0.05 → 전체 attn dead
- **per-channel**: 채널별 |α_c| < 0.05 → 그 채널 dead. attn-level dead = "전체 채널의 80% 이상이 dead".

per-channel 의 경우 추가로 **partial-dead %** (mean over channels) 도 표시.

In [ ]:
import math
import statistics

PARTIAL_DEAD_THRESH = 0.80  # 전체 채널의 80% 이상 dead → attn-level dead

print(
    f"{'α_type':>11}  {'seed':>5}  {'n_added':>7}  {'attn-dead%':>10}  "
    f"{'ch-dead%':>9}  {'mean|α|':>8}  {'final_loss':>11}"
)
print("-" * 80)
table = {}  # {alpha_type: {seed: {...stats...}}}
for alpha_type in ALPHA_TYPES:
    table[alpha_type] = {}
    for seed in SEEDS:
        r = runs[(alpha_type, seed)]
        added = [(bi, ai, a) for bi, ai, a in r["final_alphas"] if ai >= BACKBONE["n_init_attn"]]
        n_added = len(added)
        if n_added == 0:
            attn_dead_pct = math.nan
            ch_dead_pct = math.nan
            mean_abs = math.nan
        elif alpha_type == "scalar":
            # added entry: (bi, ai, tensor shape ())
            abs_vals = [a.abs().item() for _, _, a in added]
            attn_dead_pct = sum(1 for v in abs_vals if v < DEAD_THRESHOLD) / n_added
            ch_dead_pct = attn_dead_pct  # scalar 는 채널 단위 없음 = 같은 값
            mean_abs = statistics.mean(abs_vals)
        else:
            # per_channel: (bi, ai, tensor shape (hidden_dim,))
            ch_dead_per_attn = []
            attn_dead_count = 0
            all_abs = []
            for _, _, a in added:
                abs_v = a.abs()
                ch_dead = (abs_v < DEAD_THRESHOLD).float().mean().item()
                ch_dead_per_attn.append(ch_dead)
                if ch_dead >= PARTIAL_DEAD_THRESH:
                    attn_dead_count += 1
                all_abs.extend(abs_v.tolist())
            attn_dead_pct = attn_dead_count / n_added
            ch_dead_pct = statistics.mean(ch_dead_per_attn)
            mean_abs = statistics.mean(all_abs)

        n_last = min(100, len(r["losses"]))
        final_loss = sum(r["losses"][-n_last:]) / n_last if n_last > 0 else 0.0
        table[alpha_type][seed] = dict(
            n_added=n_added,
            attn_dead_pct=attn_dead_pct,
            ch_dead_pct=ch_dead_pct,
            mean_abs=mean_abs,
            final_loss=final_loss,
        )
        if n_added == 0:
            attn_str, ch_str, mean_str = "    n/a ", "    n/a ", "    n/a "
        else:
            attn_str = f"{attn_dead_pct:>10.1%}"
            ch_str = f"{ch_dead_pct:>9.1%}"
            mean_str = f"{mean_abs:>8.4f}"
        print(
            f"{alpha_type:>11}  {seed:>5}  {n_added:>7d}  {attn_str}  {ch_str}  "
            f"{mean_str}  {final_loss:>11.4f}"
        )

## 6. α_type 별 시드 invariance + 비교 verdict

In [ ]:
print(f"{'α_type':>11}  {'attn-dead range':>17}  {'mean final_loss':>16}  {'σ':>7}  verdict")
print("-" * 85)
agg = {}
for alpha_type in ALPHA_TYPES:
    rows = [table[alpha_type][s] for s in SEEDS]
    n_addeds = [r["n_added"] for r in rows]
    attn_deads_valid = [r["attn_dead_pct"] for r in rows if r["n_added"] > 0]
    final_losses = [r["final_loss"] for r in rows]
    fl_mean = statistics.mean(final_losses)
    fl_sigma = statistics.stdev(final_losses) if len(final_losses) > 1 else 0
    agg[alpha_type] = dict(mean=fl_mean, sigma=fl_sigma)
    if min(n_addeds) == 0:
        n_no_grow = sum(1 for n in n_addeds if n == 0)
        verdict = f"⚠️ trigger 미발생 ({n_no_grow}/{len(SEEDS)} seed)"
        dead_range = "n/a"
    else:
        dead_range = f"{min(attn_deads_valid):.0%} - {max(attn_deads_valid):.0%}"
        if max(attn_deads_valid) == 0:
            verdict = "✅ 모든 시드 attn dead 회피"
        elif min(attn_deads_valid) == 1.0:
            verdict = "❌ 모든 시드 attn dead"
        else:
            verdict = "⚠️ 일부 시드 attn dead"
    print(f"{alpha_type:>11}  {dead_range:>17}  {fl_mean:>16.4f}  {fl_sigma:>7.4f}  {verdict}")

# scalar vs per-channel 직접 비교
print()
print("=== scalar vs per-channel direct comparison ===")
diff = agg["scalar"]["mean"] - agg["per_channel"]["mean"]
better = "per-channel 우위" if diff > 0 else "scalar 우위" if diff < 0 else "tie"
print(f"  scalar      mean ± σ : {agg['scalar']['mean']:.4f} ± {agg['scalar']['sigma']:.4f}")
print(
    f"  per-channel mean ± σ : {agg['per_channel']['mean']:.4f} ± {agg['per_channel']['sigma']:.4f}"
)
print(f"  차이                  : {diff:+.4f}  ({better})")
n_pc_better = sum(
    1 for s in SEEDS if table["per_channel"][s]["final_loss"] < table["scalar"][s]["final_loss"]
)
print(f"  per-channel < scalar : {n_pc_better}/{len(SEEDS)} 시드")

## 7. per-channel α 채널 분포 분석

added attn 별 |α_c| histogram + partial-dead 채널 비율. scalar 의 경우 모든 채널이 같은 값
(single bar) 이라 비교 의미 없음 → per-channel 만 분석.

In [ ]:
per_ch_runs = {s: runs[("per_channel", s)] for s in SEEDS}
print(
    f"{'seed':>5}  {'added':>5}  {'block':>5}  {'attn_idx':>8}  "
    f"{'min|α|':>8}  {'mean|α|':>8}  {'max|α|':>8}  {'dead%':>6}"
)
print("-" * 75)
for seed in SEEDS:
    r = per_ch_runs[seed]
    added = [(bi, ai, a) for bi, ai, a in r["final_alphas"] if ai >= BACKBONE["n_init_attn"]]
    for idx, (bi, ai, a) in enumerate(added):
        absv = a.abs()
        dead_frac = (absv < DEAD_THRESHOLD).float().mean().item()
        print(
            f"{seed:>5}  {idx:>5}  {bi:>5}  {ai:>8}  "
            f"{absv.min().item():>8.4f}  {absv.mean().item():>8.4f}  "
            f"{absv.max().item():>8.4f}  {dead_frac:>6.1%}"
        )

## 8. 시각화 — final_loss 비교 + per-channel α distribution

In [ ]:
import matplotlib.pyplot as plt

fig = plt.figure(figsize=(14, 8))
gs = fig.add_gridspec(2, 2, height_ratios=[1, 1])

# (a) final_loss: α_type × seed bar
ax1 = fig.add_subplot(gs[0, 0])
width = 0.35
x_seeds = list(range(len(SEEDS)))
all_final = [table[t][s]["final_loss"] for t in ALPHA_TYPES for s in SEEDS]
for i, alpha_type in enumerate(ALPHA_TYPES):
    offsets = [x + (i - 0.5) * width for x in x_seeds]
    vals = [table[alpha_type][s]["final_loss"] for s in SEEDS]
    ax1.bar(offsets, vals, width=width, label=alpha_type, alpha=0.85)
ax1.axhline(
    GRAPHLM_PHASE1_4L_FINAL_LOSS, color="red", linestyle="--", lw=1, label="GraphLM 4L baseline"
)
ax1.axhline(PHASE3_SCALAR_010_MEAN, color="gray", linestyle=":", lw=1, label="Phase 3 mean")
ax1.set_xlabel("seed")
ax1.set_xticks(x_seeds)
ax1.set_xticklabels([str(s) for s in SEEDS])
ax1.set_ylabel("final loss (last 100 avg)")
ax1.set_title("final loss: α_type × seed (α_init=0.10 fixed)")
ax1.legend(loc="upper right", fontsize=8)
ax1.grid(alpha=0.3, axis="y")
ymin_d = min(min(all_final), GRAPHLM_PHASE1_4L_FINAL_LOSS, PHASE3_SCALAR_010_MEAN)
ymax_d = max(max(all_final), GRAPHLM_PHASE1_4L_FINAL_LOSS, PHASE3_SCALAR_010_MEAN)
margin = max(0.02, (ymax_d - ymin_d) * 0.15)
ax1.set_ylim(ymin_d - margin, ymax_d + margin)

# (b) attn-dead % vs ch-dead %
ax2 = fig.add_subplot(gs[0, 1])
metrics = ["attn_dead_pct", "ch_dead_pct"]
colors = ["#1f77b4", "#ff7f0e"]
labels = ["attn-dead% (per_channel)", "ch-dead% (per_channel)"]
for j, metric in enumerate(metrics):
    vals = [table["per_channel"][s][metric] * 100 for s in SEEDS]
    offsets = [x + (j - 0.5) * width for x in x_seeds]
    ax2.bar(offsets, vals, width=width, color=colors[j], label=labels[j], alpha=0.85)
ax2.set_xlabel("seed")
ax2.set_xticks(x_seeds)
ax2.set_xticklabels([str(s) for s in SEEDS])
ax2.set_ylabel("dead %")
ax2.set_title("per-channel α: attn-level vs channel-level dead %")
ax2.legend(loc="upper right", fontsize=8)
ax2.grid(alpha=0.3, axis="y")
ax2.set_ylim(
    0, max(110, max([table["per_channel"][s]["ch_dead_pct"] * 100 for s in SEEDS] + [10]) * 1.1)
)

# (c, d) seed 42 의 per-channel |α| histogram (added attn 별 subplot)
ax3 = fig.add_subplot(gs[1, :])
seed_focus = SEEDS[0]
r = per_ch_runs[seed_focus]
added = [(bi, ai, a) for bi, ai, a in r["final_alphas"] if ai >= BACKBONE["n_init_attn"]]
for idx, (bi, ai, a) in enumerate(added):
    ax3.hist(
        a.abs().numpy(),
        bins=30,
        alpha=0.5,
        label=f"added#{idx} block={bi} attn_idx={ai}",
    )
ax3.axvline(
    DEAD_THRESHOLD, color="red", linestyle="--", lw=1, label=f"dead thresh={DEAD_THRESHOLD}"
)
ax3.axvline(ALPHA_INIT, color="green", linestyle=":", lw=1, label=f"α_init={ALPHA_INIT}")
ax3.set_xlabel("|α_channel|")
ax3.set_ylabel("count of channels")
ax3.set_title(f"per-channel α histogram (seed={seed_focus}, added attn 별)")
ax3.legend(loc="upper right", fontsize=8)
ax3.grid(alpha=0.3, axis="y")

fig.tight_layout()
fig.savefig(OUT_DIR / "per_channel_alpha.png", dpi=120)
plt.show()

## 결과 요약 / Phase 5 권장 방향

이 노트북에서 확인할 것:
- §6 의 verdict — per-channel 이 scalar 대비 final_loss 개선?
- §7 의 채널 분포 — partial-dead (예: 30% 채널만 dead, 나머지 active) 패턴 존재?
- §8 (c) — |α_c| histogram 의 분산이 init 0.10 근처에서 얼마나 퍼졌나? (학습 신호 강도 측정)

**판정 시나리오**:
- **A. per-channel 명확 우위** ⭐
  - Phase 5: relative-position 함수형 α (Notion Phase 5+ 후보 참조)
  - 또는 더 fine-grained — per-head 또는 channel-grouping
- **B. per-channel ≈ scalar** (loss 차이 미미)
  - 채널 단위 표현력이 char-LM task 에 잉여 → 더 큰/복잡한 task 에서 재검증
  - 또는 full matrix α / LoRA-style low-rank ΔW 검토
- **C. per-channel 열위 / 불안정**
  - 채널 간 분산이 학습 noise 로 작용 → α scheduler / regularization 도입
  - scalar 회귀 + 다른 방향 (예: Phase 5+ 후보의 nn.Parameter 정적 shape 초월) 우선

**참고 자료**:
- Phase 5+ 후보 (Notion): https://www.notion.so/36be8b70b7aa8142bebdc9706c64ed52
- Phase 3 결과 비교 baseline: 1.7763 ± 0.0070 (scalar α=0.10, PR #46)